# 01 — Exploracao do Grafo de Entidades com NetworkX

Neste notebook exploramos o grafo de entidades do **Geolytics Dictionary** usando a biblioteca [NetworkX](https://networkx.org/). O grafo contem 107 nos e 127 arestas representando conceitos do dominio de Exploracao & Producao (E&P) de petroleo e gas no Brasil.

**Fonte de dados:** `entity-graph.json` publicado via GitHub Pages.

## Pre-requisitos

```bash
pip install networkx matplotlib requests
```

Nao e necessaria nenhuma chave de API.

## Objetivo

Ao final deste notebook voce sera capaz de:

1. Carregar o `entity-graph.json` diretamente do GitHub Pages.
2. Construir um `MultiDiGraph` NetworkX com atributos de no e aresta.
3. Calcular estatisticas basicas (nos, arestas, distribuicao de tipos).
4. Visualizar um sub-grafo ao redor do no `poco`.
5. Calcular o caminho mais curto entre `poco` e `anp`.
6. Detectar comunidades com o algoritmo de modularidade gulosa.
7. Exportar o grafo para GraphML (compativel com Gephi/yEd).

## 1. Importacoes

In [ ]:
import json
import requests
import networkx as nx
import matplotlib
import matplotlib.pyplot as plt
import collections

print('networkx', nx.__version__)
print('matplotlib', matplotlib.__version__)
# Saida esperada:
# networkx 3.x
# matplotlib 3.x

## 2. Carregamento dos dados

In [ ]:
# URL publica via GitHub Pages
GRAPH_URL = (
    'https://thiagoflc.github.io/geolytics-dictionary/data/entity-graph.json'
)

# Fallback: arquivo local (util se estiver offline)
import os
LOCAL_PATH = os.path.join(
    os.path.dirname(os.getcwd()), 'data', 'entity-graph.json'
)

try:
    resp = requests.get(GRAPH_URL, timeout=10)
    resp.raise_for_status()
    graph_data = resp.json()
    print('Dados carregados via HTTP')
except Exception as exc:
    print(f'HTTP falhou ({exc}), usando arquivo local...')
    with open(LOCAL_PATH, encoding='utf-8') as fh:
        graph_data = json.load(fh)
    print('Dados carregados do disco')

nodes = graph_data['nodes']
edges = graph_data.get('edges', [])
print(f'Nos: {len(nodes)}  Arestas: {len(edges)}')
# Saida esperada:
# Dados carregados via HTTP
# Nos: 107  Arestas: 127

## 3. Construcao do MultiDiGraph

In [ ]:
G = nx.MultiDiGraph()

# Adiciona nos com atributos
for n in nodes:
    G.add_node(
        n['id'],
        label=n.get('label', n['id']),
        label_en=n.get('label_en', ''),
        node_type=n.get('type', 'unknown'),
        color=n.get('color', '#888888'),
        size=n.get('size', 10),
    )

# Adiciona arestas com atributos
for e in edges:
    G.add_edge(
        e['source'],
        e['target'],
        relation=e.get('relation', ''),
        label_pt=e.get('relation_label_pt', ''),
        style=e.get('style', 'solid'),
    )

print(f'Nos no grafo  : {G.number_of_nodes()}')
print(f'Arestas no grafo: {G.number_of_edges()}')
print(f'E direcionado : {G.is_directed()}')
print(f'E multi-aresta: {G.is_multigraph()}')
# Saida esperada:
# Nos no grafo  : 107
# Arestas no grafo: 127
# E direcionado : True
# E multi-aresta: True

## 4. Estatisticas do grafo

In [ ]:
# Distribuicao de tipos de nos
type_counter = collections.Counter(
    data['node_type'] for _, data in G.nodes(data=True)
)
print('Distribuicao de tipos:')
for tipo, count in sorted(type_counter.items(), key=lambda x: -x[1]):
    print(f'  {tipo:<20} {count}')

# Grau medio
avg_deg = sum(d for _, d in G.degree()) / G.number_of_nodes()
print(f'\nGrau medio (total)  : {avg_deg:.2f}')

# Top-5 nos por grau de entrada (mais referenciados)
in_degrees = sorted(G.in_degree(), key=lambda x: -x[1])
print('\nTop-5 nos por grau de entrada:')
for node_id, deg in in_degrees[:5]:
    label = G.nodes[node_id].get('label', node_id)
    print(f'  {label:<30} {deg}')
# Saida esperada (aprox.):
# Distribuicao de tipos:
#   analytical           25
#   operational          17
#   ...
# Grau medio: ~2.38
# Top-5 por entrada: poco, bloco, bacia-sedimentar...

### Histograma de graus

O histograma abaixo mostra quantos nos possuem cada valor de grau total (entrada + saida). Grafos de conhecimento tipicamente exibem distribuicao de lei de potencia.

In [ ]:
degrees = [d for _, d in G.degree()]

fig, ax = plt.subplots(figsize=(7, 3))
ax.hist(degrees, bins=range(0, max(degrees) + 2), color='#378ADD', edgecolor='white')
ax.set_xlabel('Grau total (entrada + saida)')
ax.set_ylabel('Numero de nos')
ax.set_title('Distribuicao de Graus — entity-graph.json')
plt.tight_layout()
plt.show()
# Saida esperada: histograma com cauda longa a direita

## 5. Sub-grafo ao redor de `poco`

In [ ]:
CENTER = 'poco'
RADIUS = 2  # saltos ao redor do centro

# Nos a distancia <= RADIUS (tanto predecessores quanto sucessores)
neighbors = {CENTER}
current = {CENTER}
for _ in range(RADIUS):
    nxt = set()
    for n in current:
        nxt.update(G.successors(n))
        nxt.update(G.predecessors(n))
    neighbors.update(nxt)
    current = nxt

sub = G.subgraph(neighbors).copy()
print(f'Sub-grafo: {sub.number_of_nodes()} nos, {sub.number_of_edges()} arestas')
# Saida esperada:
# Sub-grafo: ~20 nos, ~25 arestas

In [ ]:
# Mapeamento tipo -> cor para visualizacao
TYPE_COLORS = {
    'operational': '#378ADD',
    'geological':  '#4CAF50',
    'contractual': '#F4A522',
    'actor':       '#D85A30',
    'instrument':  '#9C27B0',
    'equipment':   '#00BCD4',
    'analytical':  '#795548',
    'unknown':     '#888888',
}

node_colors = [
    TYPE_COLORS.get(sub.nodes[n].get('node_type', 'unknown'), '#888888')
    for n in sub.nodes()
]
node_sizes = [
    sub.nodes[n].get('size', 10) * 40 for n in sub.nodes()
]
labels = {n: sub.nodes[n].get('label', n) for n in sub.nodes()}

fig, ax = plt.subplots(figsize=(12, 8))
pos = nx.spring_layout(sub, seed=42, k=1.2)
nx.draw_networkx(
    sub, pos=pos,
    labels=labels,
    node_color=node_colors,
    node_size=node_sizes,
    edge_color='#cccccc',
    font_size=8,
    arrows=True,
    arrowsize=12,
    ax=ax,
)
ax.set_title(f'Sub-grafo raio {RADIUS} ao redor de "{CENTER}"', fontsize=13)
ax.axis('off')
plt.tight_layout()
plt.show()
# Saida esperada: grafo com poco no centro, nos vizinhos como bloco,
# bacia-sedimentar, anp, operador, etc.

## 6. Caminho mais curto entre `poco` e `anp`

In [ ]:
SOURCE = 'poco'
TARGET = 'anp'

# NetworkX trabalha com MultiDiGraph; usamos o grafo subjacente nao-dirigido
# para encontrar qualquer caminho semantico
G_undirected = G.to_undirected()

try:
    path = nx.shortest_path(G_undirected, source=SOURCE, target=TARGET)
    print(f'Caminho mais curto ({SOURCE} -> {TARGET}):')
    for i, node_id in enumerate(path):
        label = G.nodes[node_id].get('label', node_id)
        prefix = '  ' + '  ' * i + ('-> ' if i > 0 else '   ')
        print(f'{prefix}{label} ({node_id})')
    print(f'\nComprimento: {len(path) - 1} aresta(s)')
except nx.NetworkXNoPath:
    print(f'Nenhum caminho encontrado entre {SOURCE} e {TARGET}')
# Saida esperada (aprox.):
# Caminho mais curto (poco -> anp):
#    Poco (poco)
#    -> Bloco (bloco)
#    -> ANP (anp)
# Comprimento: 2 aresta(s)

## 7. Deteccao de comunidades (modularidade gulosa)

In [ ]:
# Para modularidade gulosa o grafo precisa ser nao-dirigido e conectado.
# Trabalhamos com o maior componente conectado.
from networkx.algorithms.community import greedy_modularity_communities

G_undi = G.to_undirected()
largest_cc = G_undi.subgraph(
    max(nx.connected_components(G_undi), key=len)
).copy()

communities = list(greedy_modularity_communities(largest_cc))
print(f'Numero de comunidades detectadas: {len(communities)}')

for i, comm in enumerate(communities):
    labels = [G.nodes[n].get('label', n) for n in sorted(comm)]
    print(f'\nComunidade {i+1} ({len(comm)} nos):')
    print('  ' + ', '.join(labels[:10]) + (f'  ... (+{len(labels)-10} mais)' if len(labels) > 10 else ''))
# Saida esperada:
# Numero de comunidades detectadas: ~5
# Comunidade 1: nos do dominio operacional/contratual
# Comunidade 2: nos geologicos/analiticos
# ...

## 8. Exportacao para GraphML (Gephi / yEd)

In [ ]:
# GraphML e o formato padrao para importar em Gephi, yEd, Cytoscape etc.
# Exportamos o grafo completo (nao apenas o sub-grafo).
import os

OUTPUT_PATH = os.path.join(os.path.dirname(os.getcwd()), 'notebooks', 'entity-graph.graphml')

# MultiDiGraph -> DiGraph para remover multi-arestas (GraphML nao as suporta bem)
G_simple = nx.DiGraph(G)

nx.write_graphml(G_simple, OUTPUT_PATH)
print(f'Grafo exportado para: {OUTPUT_PATH}')
print(f'  Nos  : {G_simple.number_of_nodes()}')
print(f'  Arestas: {G_simple.number_of_edges()}')
print('Abra o arquivo no Gephi (File > Open) para visualizacao interativa.')
# Saida esperada:
# Grafo exportado para: .../notebooks/entity-graph.graphml
# Nos  : 107
# Arestas: 127

## Proximo passo

Veja o notebook **02_rag_with_validator.ipynb** para aprender como construir um pipeline RAG sobre o corpus Geolytics e validar semanticamente as respostas.